# Neural Network Training with Metaheuristics (Cleaned)

## Purpose
Demonstrate optimizing MLP weights with HPPSO / PSO variants on the diabetes dataset.

## Settings
- Small network, short training — suitable for teaching, not full paper experiments.
- Paper NN experiments used multiple datasets and longer budgets (see original notebook).


## 1. Imports and configuration


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from hppso.algorithms import HPPSO, PSO
from hppso.nn.simple_mlp import SimpleNeuralNetwork, mean_squared_error, nn_objective_function

plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": True})

POP_SIZE = 30
MAX_ITERS = 100
BOUNDS = (-2, 2)
RANDOM_SEED = 42


## 2. Prepare data


In [ ]:
np.random.seed(RANDOM_SEED)
data = load_diabetes()
X = StandardScaler().fit_transform(data.data)
y = data.target.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
print(f"Train {X_train.shape}, test {X_test.shape}")


## 3. Training helper


In [ ]:
def train_mlp(algorithm: str, pop: int = POP_SIZE, iters: int = MAX_ITERS):
    """Train one MLP with a given optimizer; returns train MSE, test MSE, history."""
    nn = SimpleNeuralNetwork(X_train.shape[1], 16, 8, 1)
    n_weights = len(nn.get_weights_flat())
    objective = lambda w: nn_objective_function(w, nn, X_train, y_train)

    if algorithm == "HPPSO":
        opt = HPPSO(objective, n_pop=pop, dimensions=n_weights, max_it=iters, bounds=BOUNDS)
        train_mse, history = opt.optimize()
        nn.set_weights_flat(opt.get_best_position())
    else:
        kwargs = {}
        if algorithm == "PSO-m":
            kwargs = {"mutation_rate": 0.05, "gaussian_mutation_strength": 0.1}
        if algorithm == "PSO-RIW":
            kwargs = {"w_random_range": (0.4, 0.9), "mutation_rate": 0}
        pso = PSO(objective, [(BOUNDS[0], BOUNDS[1])] * n_weights, pop, iters, **kwargs)
        weights, train_mse, history = pso.optimize()
        nn.set_weights_flat(weights)

    test_mse = float(mean_squared_error(y_test, nn.forward(X_test)))
    return float(train_mse), test_mse, history


## 4. Compare algorithms


In [ ]:
import pandas as pd

results = []
for algo in ["PSO", "PSO-m", "PSO-RIW", "HPPSO"]:
    train_mse, test_mse, _ = train_mlp(algo)
    results.append({"Algorithm": algo, "Train MSE": train_mse, "Test MSE": test_mse})
    print(f"{algo:8s} | train={train_mse:.4f} | test={test_mse:.4f}")

pd.DataFrame(results)


## 5. Optional convergence plot (HPPSO)


In [ ]:
_, _, hist = train_mlp("HPPSO")
plt.figure()
plt.plot(hist)
plt.yscale("log")
plt.title("HPPSO training MSE on diabetes")
plt.xlabel("Iteration")
plt.ylabel("MSE")
plt.grid(True, alpha=0.35)
